In [ ]:
import os
import torch
import warnings
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import pipeline

# Ocultar avisos chatos do terminal
warnings.filterwarnings("ignore")

# 1. Detetar a Placa Gráfica (RTX 4060)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Motor de Processamento: {dispositivo.upper()}")

# 2. Carregar os PDFs
pasta_pdfs = "../pdfs_teste"
if not os.path.exists(pasta_pdfs):
    os.makedirs(pasta_pdfs)
    print(f"Criei a pasta '{pasta_pdfs}'. Por favor, coloca lá um PDF e volta a correr a célula!")
else:
    loader = DirectoryLoader(pasta_pdfs, glob="**/*.pdf", loader_cls=PDFPlumberLoader)
    documentos = loader.load()
    
    if len(documentos) == 0:
        print(f"A pasta '{pasta_pdfs}' está vazia. Coloca lá um PDF para testarmos!")
    else:
        print(f"Sucesso: Foram carregadas {len(documentos)} páginas de PDF.")

        # 3. Partir o texto do PDF em blocos mais pequenos
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
        textos_partidos = text_splitter.split_documents(documentos)

        # 4. Criar a Base de Dados Vetorial (A usar a RTX 4060!)
        print("A construir a base de conhecimento vetorial...")
        embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': dispositivo}
        )
        base_dados_vetorial = FAISS.from_documents(textos_partidos, embeddings)
        retriever = base_dados_vetorial.as_retriever(search_kwargs={"k": 6})

        # 5. Carregar o Qwen2.5 3B (O modelo mais obediente do Hugging Face)
        model_id = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" 
        
        print("A carregar o Qwen2.5 3B")

        gerador_texto = pipeline(
            "text-generation",
            model=model_id,
            model_kwargs={
                "torch_dtype": torch.float16,
                "low_cpu_mem_usage": True
            },
            max_new_tokens=256,
            temperature=1.2,
            do_sample=True,
            repetition_penalty=1.1,
            device_map="auto" 
        )
        llm_local = HuggingFacePipeline(pipeline=gerador_texto)


        # 6. A TUA PERGUNTA AO PDF (Altera este texto para algo que faça sentido com o teu PDF)
        pergunta = input("Escreve a tua pergunta sobre o conteúdo do PDF: ")
        # O sistema pesquisa no FAISS e junta o contexto
        documentos_relevantes = retriever.invoke(pergunta)
        contexto = "\n".join([doc.page_content for doc in documentos_relevantes])

        print(f"\n--- [DEBUG] O QUE A IA LEU DO PDF ---\n{contexto}\n--------------------------------------\n")
        
        # O sistema pesquisa no FAISS e junta o contexto
        documentos_relevantes = retriever.invoke(pergunta)
        contexto = "\n".join([doc.page_content for doc in documentos_relevantes])

       # 7. Prompt de Extração Multi-Documento Otimizado
        prompt = f"""<|im_start|>system
És um assistente de gestão académica rigoroso. Tens de extrair dados de dois documentos diferentes: o Cartão de Cidadão e o Currículo.

ESTRUTURA DE RESPOSTA OBRIGATÓRIA:
1. Começa sempre pelos DADOS DE IDENTIFICAÇÃO (NIF, NISS) que estão no Cartão de Cidadão.
2. Segue para os DADOS ACADÉMICOS (Curso, Datas, Instituição) que estão no Currículo.
3. Se te pedirem datas, indica 'Início: [data]' e 'Fim: [data]' para não repetires a mesma data em várias linhas.

REGRAS:
- NIF: 9 dígitos associados a 'fiscal'.
- NISS: 11 dígitos associados a 'segurança'.
- Sigla: Usa sempre 'ESTGL'.<|im_end|>
<|im_start|>user
CONTEXTO:
{contexto}

PERGUNTA: {pergunta}<|im_end|>
<|im_start|>assistant
"""

        print(f"Pergunta: {pergunta}")
        resposta = llm_local.invoke(prompt)
        
        # Limpeza para Qwen
        resposta_limpa = resposta.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
        print(f"\n--- RESULTADO DA EXTRAÇÃO ---\n{resposta_limpa}")

Motor de Processamento: CUDA
Sucesso: Foram carregadas 2 páginas de PDF.
A construir a base de conhecimento vetorial...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ A carregar o Qwen2.5 3B (Inteligente e sem filtros de privacidade)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


--- [DEBUG] O QUE A IA LEU DO PDF ---
Cartão de Cidadão
Apelido(s)
Revés Monteiro
Nome(s)
Tomás David
Data de
Sexo Altura Nacionalidade
Nascimento
Masculino 179 PT 04-07-2002
Nº cartão Data de validade
30800941 0ZX7 12-01-2027
Filiação Nº identificação Nº segurança Nº utente saúde
fiscal social
Dinis de Carvalho Monteiro 274244594 12032335359 361815808
Elisabete Gonçalves Revés Monteiro
Assinatura do titular
2023.07.23 21:44:55 GMT+01:00 1
Tomás Monteiro
Autorização de trabalho: Portuguesa Data de nascimento: 04/07/2002
Local de nascimento: Lisboa, Portugal Nacionalidade: Portuguesa Número de telemóvel:
(+351) 934023227 (Telemóvel) Endereço de email: tsmonteiro17@icloud.com
Endereço: Avenida de Ceuta 36, 2700-190, Lisboa, Portugal (Casa)
EXPERIÊNCIA PROFISSIONAL
GOODS FLOW CO-WORKER - LOGISTICS – IKEA – 23/06/2022 – 07/01/2023 – LISBOA, PORTUGAL
OPERADOR DE CAIXA – PINGO DOCE – 14/11/2021 – 12/05/2022 – CANEÇAS, PORTUGAL
EDUCAÇÃO E FORMAÇÃO
18/10/2023 – ATUAL Lamego, Viseu, Portugal
L